# 2. Data Cleaning & Preprocessing

This notebook demonstrates the data preprocessing pipeline: handling missing values, outlier detection, and feature normalization.

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import sys
import os

# Add src to path
sys.path.insert(0, '../')
from src.data.loaders import load_raw_data, save_processed_data
from src.data.processors import DataProcessor

print('Libraries and modules loaded')

## Load Raw Data

In [ ]:
# Load raw data
df_raw = load_raw_data('../data/raw/alzheimers_biomarkers.csv')
print(f'Raw data shape: {df_raw.shape}')
print(f'\nFirst few rows:')
print(df_raw.head())

## Step 1: Handle Missing Values

In [ ]:
# Check for missing values
print('Missing values before cleaning:')
print(df_raw.isnull().sum())
print(f'\nTotal missing: {df_raw.isnull().sum().sum()}')

# Initialize processor
processor = DataProcessor()

# Handle missing values
df_clean = processor.handle_missing_values(df_raw)
print(f'\nShape after removing missing values: {df_clean.shape}')

## Step 2: Outlier Detection

In [ ]:
# Detect outliers using IQR method
df_with_outlier_info, outlier_stats = processor.detect_outliers(df_clean)

print('Outlier Detection Results (IQR method with threshold=1.5):')
print('-' * 70)
for column, stats_dict in outlier_stats.items():
    print(f'{column:20} | Lower: {stats_dict["lower_bound"]:8.2f} | Upper: {stats_dict["upper_bound"]:8.2f} | Outliers: {stats_dict["n_outliers"]}')

## Visualize Outliers

In [ ]:
# Box plots to visualize outliers
features = ['p_tau', 'np_tau', 'ab_42', 'age', 'years_education', 'mmse']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, feature in enumerate(features):
    axes[idx].boxplot(df_clean[feature])
    axes[idx].set_ylabel(feature)
    axes[idx].set_title(f'Box plot of {feature}')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: Feature Normalization

In [ ]:
# Separate features and target
X = df_clean[features].copy()
y = df_clean['amyloid_positive'].copy()

print('Before normalization:')
print(f'Feature means: {X.mean().round(3)}')
print(f'Feature stds: {X.std().round(3)}')

# Normalize
X_normalized, _ = processor.normalize_features(X, fit=True)

print('\nAfter normalization:')
print(f'Feature means: {X_normalized.mean(axis=0).round(10)}')
print(f'Feature stds: {X_normalized.std(axis=0).round(3)}')

## Visualize Before/After Normalization

In [ ]:
# Compare distributions before and after
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, feature in enumerate(features):
    axes[idx].hist(X[feature], bins=30, alpha=0.6, label='Before', color='blue', edgecolor='black')
    axes[idx].hist(X_normalized[:, idx], bins=30, alpha=0.6, label='After', color='red', edgecolor='black')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'Distribution before/after normalization')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Full Processing Pipeline

In [ ]:
# Complete preprocessing pipeline
df_processed = processor.process(df_raw, fit_scaler=True)

print(f'Processed data shape: {df_processed.shape}')
print(f'\nProcessed data (first 5 rows):')
print(df_processed.head())

print(f'\nProcessed data statistics:')
print(df_processed.describe())

## Save Processed Data

In [ ]:
# Save processed data
output_path = '../data/processed/alzheimers_processed.csv'
save_processed_data(df_processed, output_path)

print(f'Processed data saved to: {output_path}')

# Verify
df_check = pd.read_csv(output_path)
print(f'Verification - shape: {df_check.shape}')

## Summary of Preprocessing Steps

1. **Missing Values**: Removed rows with NaN values (0 rows in this dataset)
2. **Outlier Detection**: Identified outliers using IQR method (1.5 × IQR threshold)
3. **Feature Normalization**: Standardized features to zero mean and unit variance using StandardScaler
4. **Data Quality**: Verified all features are in expected ranges after preprocessing

The processed data is now ready for feature engineering and model training.